# TextRank keyword extraction

Find important words and phrases in a paragraph.


## Input

Change the text below to try another paragraph.


In [ ]:
# Text to process.
Text = "Compatibility of systems of linear constraints over the set of natural numbers. \
Criteria of compatibility of a system of linear Diophantine equations, strict inequations, and \
nonstrict inequations are considered. \
Upper bounds for components of a minimal set of solutions and \
algorithms of construction of minimal generating sets of solutions for all \
types of systems are given. \
These criteria and the corresponding algorithms for constructing \
a minimal supporting set of solutions can be used in solving all the \
considered types of systems and systems of mixed types."


## Clean the text

Use lowercase letters and split the text into words.


In [ ]:
# Clean and split the text.
import nltk
from nltk import word_tokenize
import string


def clean(text):
    text = text.lower()
    printable = set(string.printable)
    text = filter(lambda x: x in printable, text)
    text = "".join(list(text))
    return text

Cleaned_text = clean(Text)
text = word_tokenize(Cleaned_text)

print ("Tokenized Text: \n")
print (text)


## Tag the words

Find nouns, adjectives, verbs, and other word types.


In [ ]:
# Find word types.
POS_tag = nltk.pos_tag(text)

print ("Tokenized Text with POS tags: \n")
print (POS_tag)


## Use base words

For example, change "systems" to "system".


In [ ]:
# Convert nouns and adjectives to their base forms.
from nltk.stem import WordNetLemmatizer

wordnet_lemmatizer = WordNetLemmatizer()

adjective_tags = ['JJ','JJR','JJS']

lemmatized_text = []

for word in POS_tag:
    if word[1] in adjective_tags:
        lemmatized_text.append(str(wordnet_lemmatizer.lemmatize(word[0],pos="a")))
    else:
        lemmatized_text.append(str(wordnet_lemmatizer.lemmatize(word[0])))

print ("Text tokens after lemmatization of adjectives and nouns: \n")
print (lemmatized_text)


## Tag the base words

These tags help us choose useful words.


In [ ]:
# Tag the base words.
POS_tag = nltk.pos_tag(lemmatized_text)

print ("Lemmatized text with POS tags: \n")
print (POS_tag)


## Filter word types

Keep nouns, adjectives, gerunds, and foreign words. Remove punctuation.


In [ ]:
# Collect words and punctuation to skip.
stopwords = []

wanted_POS = ['NN','NNS','NNP','NNPS','JJ','JJR','JJS','VBG','FW']

for word in POS_tag:
    if word[1] not in wanted_POS:
        stopwords.append(word[0])

punctuations = list(str(string.punctuation))

stopwords = stopwords + punctuations


## Load common words

Read extra words to skip from long_stopwords.txt.


In [ ]:
# Load the extra stopwords.
with open("long_stopwords.txt", encoding="utf-8") as stopword_file:
    lots_of_stopwords = [line.strip() for line in stopword_file]

stopwords_plus = set(stopwords + lots_of_stopwords)


## Remove common words

Keep the words that may be useful keywords.


In [ ]:
# Remove stopwords.
processed_text = []
for word in lemmatized_text:
    if word not in stopwords_plus:
        processed_text.append(word)
print (processed_text)


## List unique words

Each word appears once in the vocabulary.


In [ ]:
# Keep each word once.
vocabulary = sorted(set(processed_text))
print (vocabulary)


## Connect nearby words

Words in the same three-word window are connected. Closer words get a stronger connection.


In [ ]:
# Connect words that appear close together.
import numpy as np
import math
vocab_len = len(vocabulary)

weighted_edge = np.zeros((vocab_len,vocab_len),dtype=np.float32)

score = np.zeros((vocab_len),dtype=np.float32)
window_size = 3
covered_coocurrences = []

for i in range(0,vocab_len):
    score[i]=1
    for j in range(0,vocab_len):
        if j==i:
            weighted_edge[i][j]=0
        else:
            for window_start in range(max(1, len(processed_text) - window_size + 1)):

                window_end = window_start+window_size

                window = processed_text[window_start:window_end]

                if (vocabulary[i] in window) and (vocabulary[j] in window):

                    index_of_i = window_start + window.index(vocabulary[i])
                    index_of_j = window_start + window.index(vocabulary[j])


                    if [index_of_i,index_of_j] not in covered_coocurrences:
                        weighted_edge[i][j]+=1/math.fabs(index_of_i-index_of_j)
                        covered_coocurrences.append([index_of_i,index_of_j])


## Add connection weights

Find the total connection weight for each word.


In [ ]:
# Add each word's connection weights.
inout = np.zeros((vocab_len),dtype=np.float32)

for i in range(0,vocab_len):
    for j in range(0,vocab_len):
        inout[i]+=weighted_edge[i][j]


## Score the words

Update each score using connected words. Stop when the scores change very little.


In [ ]:
# Repeat until scores settle.
MAX_ITERATIONS = 50
d=0.85
threshold = 0.0001

for iter in range(0,MAX_ITERATIONS):
    prev_score = np.copy(score)

    for i in range(0,vocab_len):

        summation = 0
        for j in range(0,vocab_len):
            if weighted_edge[i][j] != 0:
                summation += (weighted_edge[i][j]/inout[j])*score[j]

        score[i] = (1-d) + d*(summation)

    if np.sum(np.fabs(prev_score-score)) <= threshold:
        print("Converging at iteration "+str(iter)+"....")
        break


In [ ]:
# Show each word score.
for i in range(0,vocab_len):
    print("Score of "+vocabulary[i]+": "+str(score[i]))


## Make phrases

Split the text at common words and punctuation.


In [ ]:
# Group words into phrases.
phrases = []

phrase = " "
for word in lemmatized_text:

    if word in stopwords_plus:
        if phrase!= " ":
            phrases.append(str(phrase).strip().split())
        phrase = " "
    elif word not in stopwords_plus:
        phrase+=str(word)
        phrase+=" "

if phrase.strip():
    phrases.append(phrase.strip().split())

print("Partitioned Phrases (Candidate Keyphrases): \n")
print(phrases)


## Remove repeated phrases


In [ ]:
# Keep each phrase once.
unique_phrases = []

for phrase in phrases:
    if phrase not in unique_phrases:
        unique_phrases.append(phrase)

print("Unique Phrases (Candidate Keyphrases): \n")
print(unique_phrases)


## Remove extra single words

Skip a single word when it already belongs to a longer phrase.


In [ ]:
# Remove single words already used in longer phrases.
for word in vocabulary:
    for phrase in unique_phrases:
        if (word in phrase) and ([word] in unique_phrases) and (len(phrase)>1):
            unique_phrases.remove([word])

print("Thinned Unique Phrases (Candidate Keyphrases): \n")
print(unique_phrases)


## Score the phrases

Add the scores of the words in each phrase.


In [ ]:
# Add word scores to get phrase scores.
phrase_scores = []
keywords = []
for phrase in unique_phrases:
    phrase_score=0
    keyword = ''
    for word in phrase:
        keyword += str(word)
        keyword += " "
        phrase_score+=score[vocabulary.index(word)]
    phrase_scores.append(phrase_score)
    keywords.append(keyword.strip())

i=0
for keyword in keywords:
    print ("Keyword: '"+str(keyword)+"', Score: "+str(phrase_scores[i]))
    i+=1


## Show the keywords

Print up to ten phrases with the highest scores.


In [ ]:
# Show the highest scoring phrases.
sorted_index = np.flip(np.argsort(phrase_scores),0)

keywords_num = 10

print("Keywords:\n")

for i in range(min(keywords_num, len(keywords))):
    print(str(keywords[sorted_index[i]])+", ", end=' ')
